In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
import math
import numpy as np
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [3]:
from ggblab import GeoGebra
ggb = await GeoGebra().init(use_vscode=False)

Using local cached file: xsd/common.xsd


In [4]:
%pwd

'/Users/manabu/work/ggblab/examples'

In [7]:
%cd ggblab/examples

/Users/manabu/work/ggblab/examples


In [4]:
ggb.file.load('eg11_slider.ggb')
# ggb.file.load('2025_06_08.ggb')

In [5]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [6]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [7]:

df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
0,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,9,false,false,false
1,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,false,false
2,"""B""","""point""","""Point(c)""","""B = (-1, 0)""",null,9,false,false,false
3,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,false,false,false
4,"""n""","""numeric""",null,"""n = 0""",null,9,true,true,false
5,"""i""","""numeric""",null,"""i = 0""",null,9,true,true,false
6,"""C""","""point""","""Point(f)""","""C = (1.2960082480647, 0)""",null,9,false,false,false
7,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 1.2960082480647""",null,8,false,false,false
8,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(?, ?)}""",null,8,true,false,false


In [8]:
p = ConstructionTreeParser(df)
g1 = p.parse()

In [9]:
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    try:
        g1.nodes[n]['label'] = f"{n} ({t})"
    except:
        pass

In [10]:
nx.set_node_attributes(g1, labels_map, "label")
nx.write_network_text(g1, with_labels="label")

╟── A (point)
╎   ├─╼ c (circle)
╎   │   └─╼ B (point)
╎   │       └─╼ f (line) ╾ A (point)
╎   │           ├─╼ C (point)
╎   │           │   └─╼ g (line) ╾ f (line)
╎   │           └─╼  ...
╎   └─╼  ...
╟── l1 (list)
╎   ├─╼ D (point)
╎   └─╼ E (point)
╟── a (curvecartesian)
╎   ├─╼ G (point) ╾ b (numeric)
╎   │   ├─╼ h (line) ╾ a (curvecartesian)
╎   │   │   ├─╼ I (point) ╾ d (circle)
╎   │   │   └─╼ J (point) ╾ d (circle)
╎   │   │       └─╼ w (vector) ╾ G (point)
╎   │   │           └─╼ e (numeric) ╾ u (vector)
╎   │   ├─╼ u (vector) ╾ F (point)
╎   │   │   ├─╼ j (numeric)
╎   │   │   └─╼  ...
╎   │   ├─╼ d (circle)
╎   │   │   └─╼  ...
╎   │   ├─╼ k (numeric) ╾ F (point)
╎   │   └─╼  ...
╎   └─╼  ...
╟── b (numeric)
╎   └─╼  ...
╙── F (point)
    └─╼  ...


In [11]:
g2 = p.parse_subgraph()
nx.write_network_text(g2)

╟── b
╎   └─╼ G
╎       └─╼ h
╙── A
    └─╼ c
        ├─╼ f
        │   ├─╼ C ╾ B
        │   └─╼ g ╾ B
        └─╼ B
            └─╼  ...


In [12]:
from ggblab_extra import hungarian_similarity

In [19]:
s, r = hungarian_similarity(g1, g2)
s

0.6724137931034483

In [15]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn,DependsOn_minimal
u32,str,str,str,str,str,u32,bool,bool,bool,list[str],list[str]
0,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,9,false,false,false,[],[]
1,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,false,false,"[""A""]","[""A""]"
2,"""B""","""point""","""Point(c)""","""B = (-1, 0)""",null,9,false,false,false,"[""A"", ""c""]","[""c""]"
3,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,false,false,false,"[""A"", ""B"", ""c""]","[""B""]"
4,"""n""","""numeric""",null,"""n = 0""",null,9,true,true,false,[],[]
5,"""i""","""numeric""",null,"""i = 0""",null,9,true,true,false,[],[]
6,"""C""","""point""","""Point(f)""","""C = (1.2960082480647, 0)""",null,9,false,false,false,"[""A"", ""B"", … ""f""]","[""f""]"
7,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 1.2960082480647""",null,8,false,false,false,"[""A"", ""B"", … ""f""]","[""C"", ""f""]"
8,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(?, ?)}""",null,8,true,false,false,[],[]


In [12]:
l1 = 'n'
l2 = 'i'

In [13]:
await ggb.listen(l1, True)
await ggb.listen(l2, True)

{}

In [141]:
await ggb.listen('a', False)
await ggb.listen('b', False)

{}

In [14]:
ggb.comm.shared_objects

{'n': 'n = 0', 'i': 'i = 0'}

In [15]:
# import ipywidgets as widgets
# label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
# label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
# display(label1, label2)

In [16]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['a']
    n = int(changes[l1].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(9), [True]*n, fillvalue=False)))
    r = await ggb.function('getXML', [l2])
    o = ggb.file.ggb_schema.decode(r)
    o['value'][0]['@val'] = '0'
    x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
    r = await ggb.function('evalXML' , [x])

In [17]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [18]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['b']
    m = int(changes[l2].split()[2])
    n = int(ggb.comm.shared_objects[l1].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [19]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [104]:
ggb.comm.clear_shared_listeners()

2

In [29]:
await ggb.command('Curve(x, x^2, x, -10, 10)')

'a'

In [30]:
await ggb.command('a(b)')

'G'

In [32]:
await ggb.command('Tangent(G, a)')

'h'

In [33]:
await ggb.command('u w')

'e'

In [34]:
await ggb.command('sqrt(u u)')

'j'

In [36]:
await ggb.command('k = Distance(a,F)')

'k'

In [7]:
await ggb.command('{Tangent(c,C)}')

'l2'

In [96]:
r1 = await ggb.function('getValueString', ['l1'])
# l1 = ggb.parser.tokenize_with_commas(r1)
l2 = ggb.parser.tokenize(r1)
l2

['l1', '=', []]

In [97]:
r2 = await ggb.function('getValueString', ['l2'])
l = ggb.parser.tokenize(r2, simplify=True)
l

['l2',
 '=',
 [['-0.7453559924999',
   'x',
   '-',
   '0.8333333333333',
   'y',
   '=',
   '-1.1180339887499'],
  ['0.7453559924999',
   'x',
   '-',
   '0.8333333333333',
   'y',
   '=',
   '1.1180339887499']]]

In [52]:
await ggb.command("l1(2)")

'E'

In [50]:
await ggb.command("{Intersect[c, g]}")

'l1'

In [200]:
async def getCoords(list_points=[]):
    r = await ggb.function(["getXcoord", "getYcoord"], [[p] for p in list_points])
    arr = np.array(list(zip(*r)), dtype=float)
    arr[np.isclose(arr, 0., atol=1e-9)] = 0.
    arr = arr[~np.any(np.isnan(arr), axis=1)]
    return arr.tolist()

In [219]:
await getCoords(['D', 'E'])

[[0.8000000000000002, 0.5999999999999999], [0.7999999999999999, -0.6]]

In [222]:
r = await ggb.function("getValueString", ['l1'])
len(list(toCoords(r)))

2

In [215]:
from collections.abc import Iterable

def toCoords(r):
    # ret = []
    for e in ggb.parser.tokenize_with_commas(r):
        if isinstance(e, Iterable) and not isinstance(e, (str, bytes)):
            r2 = [float(e2) for e2 in e if e2 not in [',', '?']]
            if r2:
                # ret.append(r2)
                yield r2
    # return ret


In [223]:
import ipywidgets as widgets
label = widgets.Label(value="")
display(label)

Label(value='')

In [224]:
await ggb.listen('l1')

{}

In [225]:
ggb.comm.shared_objects

{'l1': 'l1 = {(0.8, 0.6), (0.8, -0.6)}'}

In [226]:
async def on_shared_update(changes):
    n = len(list(toCoords(changes['l1'])))
    label.value = f"Length of l1: {n}"

In [227]:
ggb.comm.add_shared_listener(on_shared_update)

True

In [6]:
%pwd

'/Users/manabu/work/ggblab'

In [ ]:
# ggb.file.source_file = 'eg11_slider.ggb'

In [39]:
ggb.file.base64_buffer = await ggb.function("getBase64")

In [40]:
ggb.file.save(overwrite=True)

* 原則（教育観）:
    - 目的化: 再現は「結果」ではなく「理解（なぜその操作か）」を目的にする。
    - 予測→検証: 次に何が起きるか予測させてから操作させる。
    - 説明要求: 手順ごとに短い理由説明（1文）を書かせる。
    - 変奏課題: パラメータを少し変えた課題で本質が移るか確認する。
    - 生成的課題: 「同じ発想で別の図形を作る」など転移を問う。
* ggblabで実装できる仕組み（短）:
    - 段階公開（layer slider）: 各レイヤーに「解説」「問い」「期待する操作」を紐付け、スライダーで段階的に提示。
    - 予測プロンプト: 各ステップの前に「次に何が起きる？」を表示し、回答を記録。
    - 説明入力欄: 学生が操作毎に短い説明を入力 → 教師や自動ルールでフィードバック。
    - 変化タスク自動化: DataFrame→コマンド生成を利用してパラメータをランダム化した派生課題を作る。
    - 操作ログ＋解析: 操作順・所要時間・試行回数をログ化して学習診断に使う。
    - 差分フィードバック: 学生構成と模範構成を比較して「次に直すべき一手」を提示。

In [29]:
await ggb.function("getVersion")

'5.2.909.9'